# 🚀 GPT

In this notebook, we'll walk through the steps required to train your own GPT model on the wine review dataset

The code is adapted from the excellent [GPT tutorial](https://keras.io/examples/generative/text_generation_with_miniature_gpt/) created by Apoorv Nandan available on the Keras website.

In [88]:
# === Colab Setup (auto-skipped outside Google Colab) ===
import sys, os, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = "yukai-yabuta/Generative_Deep_Learning_2nd_Edition"
    BRANCH = "main"
    NOTEBOOK_REL = "notebooks/09_transformer/gpt"
    REPO_DIR = "/content/repo"

    if not pathlib.Path(REPO_DIR).exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "-b", BRANCH,
             f"https://github.com/{REPO}.git", REPO_DIR],
            check=True,
        )

    pathlib.Path("/app").mkdir(exist_ok=True)
    if not pathlib.Path("/app/data").exists():
        pathlib.Path("/content/data").mkdir(exist_ok=True)
        os.symlink("/content/data", "/app/data")

    os.chdir(f"{REPO_DIR}/{NOTEBOOK_REL}")

    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    except Exception:
        print("⚠️ Set KAGGLE_USERNAME & KAGGLE_KEY via Colab Secrets (🔑 sidebar).")

    subprocess.run(["pip", "install", "-q", "kaggle"], check=True)

    wine_path = pathlib.Path("/app/data/wine-reviews/winemag-data-130k-v2.json")
    if not wine_path.exists():
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", "zynicide/wine-reviews",
             "-p", "/app/data/wine-reviews", "--unzip"],
            check=True,
        )

    import tensorflow as tf
    print(f"✅ Colab setup done.\n   cwd: {os.getcwd()}\n   TF: {tf.__version__}\n   GPUs: {tf.config.list_physical_devices('GPU')}")


✅ Colab setup done.
   cwd: /content/repo/notebooks/09_transformer/gpt
   TF: 2.20.0
   GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## このノートブックの概要

ワインレビューのデータセットを使って、デコーダ型トランスフォーマ (GPT) を学習するノートブックです。書籍 9 章の流れに沿って、トークン化 → 因果アテンション → トランスフォーマブロック → 訓練 → 生成 + アテンション可視化、と段階的に組み立てていきます。

主な構成要素:

- **トークン化**: `TextVectorization` で単語列を整数 ID 列に変換
- **位置エンコーディング**: トークン埋め込み + 位置埋め込みの加算で語順情報を保持
- **因果マスク付きマルチヘッドアテンション**: 未来のトークンを参照させない自己回帰の仕組み
- **シングルブロック構成**: 原論文の 12 ブロックではなく 1 ブロックだけの簡略版

書籍の節番号との対応:

- §9.2.1-9.2.2 → 「Load the data」 / 「Tokenize」
- §9.2.3-9.2.6 → 「Causal mask」 / 「TransformerBlock」
- §9.2.7 → 「TokenAndPositionEmbedding」
- §9.2.8 → 「Train」
- §9.2.9 → 「Generate text」 (テキスト生成 + アテンション可視化)


In [89]:
# 標準ライブラリと TensorFlow/Keras を読み込む
import numpy as np
import json
import re
import string
from IPython.display import display, HTML  # アテンション可視化で HTML を表示するため

import tensorflow as tf
from tensorflow.keras import layers, models, losses, callbacks


## 0. Parameters <a name="parameters"></a>

In [90]:
# ====================================================================
# このノートブックで使うパラメータをまとめて定義する。
# 大きく 2 種類に分かれる:
#   - 「モデルの構造」を決めるアーキテクチャ系 (EMBEDDING_DIM, KEY_DIM, N_HEADS など)
#     → モデル定義時に固定。学習中は変えられない。一度学習を始めたら構造が変わる
#       変更はできない (重みの形が違うので)
#   - 「学習の長さ・進め方」を決める学習設定系 (EPOCHS, BATCH_SIZE など)
#     → モデル構造は変えず、訓練ループの挙動だけを変える。気軽に変更可能
# どちらも数値だが、種別が違うので注意。
# ====================================================================


# --- データ・語彙の規模 ---

# 語彙サイズ。学習データに出てくる全単語のうち、頻度上位 1 万件を採用する。
# 残りはまとめて [UNK] (unknown) という 1 個の特殊トークンに置き換えられる。
# 大きいほど珍しい単語まで表現できるが、Embedding 層のパラメータ数 (= VOCAB_SIZE × EMBEDDING_DIM) も増える。
VOCAB_SIZE = 10000

# モデルが一度に見るトークン (単語) 数。80 トークン分の系列を入力として与え、
# 各位置で「次のトークン」を予測する。
# 長くするほど遠い文脈まで参照できるが、アテンション計算量が O(MAX_LEN²) で増える。
MAX_LEN = 80


# --- 埋め込み・モデルの内部次元 ---

# 1 単語あたりの表現ベクトルの長さ。例えば "wine" という単語は 256 個の数値からなる
# ベクトルに変換される。次元が大きいほど豊かなニュアンスを表現できるが、メモリと計算量が増える。
EMBEDDING_DIM = 256

# アテンションでの「キー」と「クエリ」のベクトル次元。
#
# アテンションとは: 「今見ている単語 (クエリ)」と「文中の他の単語 (キー)」の関連度を測り、
# 関連度が高い単語の情報を優先的に集める仕組み。具体的には:
#   1. 各単語の埋め込みベクトル (EMBEDDING_DIM 次元) を、Q (クエリ)・K (キー)・V (バリュー) の
#      3 種類のベクトルに線形変換する。
#   2. Q と K の内積 (= 類似度) を計算し、softmax で「アテンション重み」を得る。
#   3. その重みで V を加重平均 → 文脈情報を集約したベクトルを得る。
#
# KEY_DIM はその Q と K の次元。例えで言うと:
#   - クエリ = 「今、何を探しているか」のフィルター
#   - キー   = 「私はこういう情報を持っています」というラベル
#   - 両者の内積が大きいほど「この単語に注意を向けるべき」と判断される。
KEY_DIM = 256

# マルチヘッドアテンションのヘッド数。← アーキテクチャ系 (モデルの構造を決める)
#
# アテンションを 1 回だけやるのではなく、N_HEADS 個の独立した射影行列 (Q/K/V) を持つ
# 「ヘッド」を並列に走らせて、最後に結合する。
#
# 重要な点: 各ヘッドが何を学ぶかは事前に決められない。射影行列はランダム初期化され、
# 学習中の勾配の流れ方が違うことから (symmetry breaking)、結果的に違うパターンを
# 捉えるように収束する -- いわゆる "emergent specialization"。
#
# 学習後のアテンション重みを可視化した研究 (例: Voita et al. 2019,
# https://arxiv.org/abs/1905.09418) では、構文関係や指示語の解消、直前トークンへの
# 注目、など色々なパターンを学ぶヘッドが見つかる。
# 一方で、実態としては:
#   - 多くのヘッドが冗長 (似たようなパターンを学ぶ)
#   - 「死んでいる」ヘッド (常に均一な重みを出して貢献していない) もある
#   - 「主語-動詞」のようにきれいに解釈できるヘッドはむしろ少数派
#
# したがって「ヘッドを増やす = 違う関係を強制的に学ばせる」というより、
# 「違うパターンを学ぶ機会を与える (統計的に多様化を促す)」という理解が正確。
#
# EPOCHS との混同に注意: N_HEADS は「モデルの中に何個の並列ヘッドを置くか」という
# 空間的・構造的なパラメータ。一方 EPOCHS は「学習を何周繰り返すか」という時間的な
# 反復回数。料理教室のたとえで言うと:
#   - N_HEADS = 教室にいる生徒の数 (並列に学ぶ視点の数。教室の規模)
#   - EPOCHS  = レシピを何回繰り返し練習するか (時間軸の反復)
# まったく別の軸のチューニングなので注意。
N_HEADS = 2

# トランスフォーマブロック内にある全結合層 (FFN) の中間層のサイズ。
# アテンション層の後に、各位置で独立に通る 2 層 MLP の中間次元。
# アテンションが「位置同士のやり取り」を担うのに対し、FFN は「各位置での非線形変換」を担う。
FEED_FORWARD_DIM = 256


# --- 学習・実行制御 ---

# バリデーション用に取り分けるデータの割合 (このノートブックでは実際には未使用)。
VALIDATION_SPLIT = 0.2

# 乱数シード。再現性のため固定 (Dropout / シャッフルの乱数を毎回同じにする)。
SEED = 42

# True にすると、保存済みモデル ("./models/gpt.keras") を再ロードして学習をスキップする。
# 1 回学習を回した後、生成や可視化だけ試したいときに True にして再起動する用。
LOAD_MODEL = False

# 一度に GPU に流す系列の数。32 本のレビューを同時に処理して 1 回の勾配更新を行う。
# 大きいほど学習が安定して速いが、GPU メモリを多く使う。
BATCH_SIZE = 32

# 学習データ全体を何周するか。← 学習設定系 (訓練ループの挙動を決める)
#
# 5 エポック = 訓練データ (約 13 万件) を 5 回繰り返して学習する。
# 増やすほど学習は進むが、過学習リスクと時間 (Colab T4 で 1 エポック ≈ 数分) が増える。
#
# N_HEADS との混同に注意: EPOCHS は「データを何周するか」という時間的な反復回数。
# モデル構造は一切変わらない (重みの値だけが更新ループで変わる)。
#   - EPOCHS を 5→10 に増やす → 学習時間が単純に 2 倍。モデルの形は変わらない
#   - N_HEADS を 2→4 に増やす → モデル内のヘッドが 2 個増える。1 ステップが少し重くなる
# 一言でまとめると:
#   - N_HEADS = モデルの「形」(空間的なリソース、再構築しないと変えられない)
#   - EPOCHS  = 学習の「長さ」(時間的なリソース、気軽に増減できる)
EPOCHS = 5


## 1. Load the data <a name="load"></a>

Kaggle の `wine-reviews` データセット (約 13 万件) を読み込み、`wine review : <country> : <province> : <variety> : <description>` の形式の文字列に整形します。

冒頭に「`wine review : `」を付ける狙いは、生成時にこれをプロンプトとして与えると「ワインレビューを書く」モードでテキストが続くようにモデルを誘導できるようにすることです (書籍 9.2.9.1 で `temperature` を変えながら使う prompt と一致)。


In [91]:
# Kaggle からダウンロードしたワインレビュー JSON を読み込む (約 13 万件)
with open("/app/data/wine-reviews/winemag-data-130k-v2.json") as json_data:
    wine_data = json.load(json_data)


In [92]:
# 1 件のレビューの構造を確認 (country, province, variety, description などの dict)
wine_data[10]


{'points': '87',
 'title': 'Kirkland Signature 2011 Mountain Cuvée Cabernet Sauvignon (Napa Valley)',
 'description': 'Soft, supple plum envelopes an oaky structure in this Cabernet, supported by 15% Merlot. Coffee and chocolate complete the picture, finishing strong at the end, resulting in a value-priced wine of attractive flavor and immediate accessibility.',
 'taster_name': 'Virginie Boone',
 'taster_twitter_handle': '@vboone',
 'price': 19,
 'designation': 'Mountain Cuvée',
 'variety': 'Cabernet Sauvignon',
 'region_1': 'Napa Valley',
 'region_2': 'Napa',
 'province': 'California',
 'country': 'US',
 'winery': 'Kirkland Signature'}

In [93]:
# country / province / variety / description が揃っているレビューだけを残し、
# "wine review : <country> : <province> : <variety> : <description>" の形式に整形
filtered_data = [
    "wine review : "
    + x["country"]
    + " : "
    + x["province"]
    + " : "
    + x["variety"]
    + " : "
    + x["description"]
    for x in wine_data
    if x["country"] is not None
    and x["province"] is not None
    and x["variety"] is not None
    and x["description"] is not None
]


In [94]:
# フィルタ後の件数を確認 (NULL を含むレビューが落ちて少し減る)
n_wines = len(filtered_data)
print(f"{n_wines} recipes loaded")


129907 recipes loaded


In [95]:
# 整形済みデータの中身を 1 件覗いてみる
example = filtered_data[25]
print(example)


wine review : US : California : Pinot Noir : Oak and earth intermingle around robust aromas of wet forest floor in this vineyard-designated Pinot that hails from a high-elevation site. Small in production, it offers intense, full-bodied raspberry and blackberry steeped in smoky spice and smooth texture.


## 2. Tokenize the data <a name="tokenize"></a>

GPT は単語単位のトークンを扱うため、句読点を独立した「単語」として認識させる前処理 (`pad_punctuation`) を入れます。例えば `"great wine."` → `"great wine ."` のように記号の周りに空白を入れることで、Keras の `TextVectorization` がそれぞれ独立したトークンとして語彙に登録します。

`TextVectorization` の主な引数:

- `standardize="lower"` … 小文字化のみ (句読点は前処理済みなので除去しない)
- `max_tokens=VOCAB_SIZE` … 上位 `VOCAB_SIZE` 件だけ採用、残りは `[UNK]`
- `output_sequence_length=MAX_LEN + 1` … `+1` しているのは、後段で入力 (前 N) とターゲット (後 N) に **1 トークンずらして** 分割するため


In [96]:
# 記号類を独立した「単語」として扱えるよう、前後にスペースを入れる前処理。
#
# なぜ必要か:
#   後段の TextVectorization は「半角スペース区切りで単語に切る」ため、
#   前処理せずに "wine," を渡すと "wine," が 1 トークンとして登録されてしまい、
#   "wine" と別物扱いになる。前後にスペースを挟んで分割可能にする。
def pad_punctuation(s):
    # 1. string.punctuation = '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~' の全記号 +
    #    改行 (\n) の前後に半角スペースを挟む。
    #    パターンは ([!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~, '\n']) で、
    #    [...] は文字クラス (どれか 1 文字)、(...) はキャプチャグループ、
    #    置換側の \1 でマッチした 1 文字を参照して " <記号> " に書き換える。
    #
    #    例: "This wine, full-bodied. Great!"
    #         → "This wine ,  full - bodied .  Great ! "
    s = re.sub(f"([{string.punctuation}, '\n'])", r" \1 ", s)

    # 2. 1 行目で「元のスペース + 新たに足したスペース」が重なって
    #    "   " のように複数並ぶので、1 個のスペースに縮める。
    #    例: "This wine ,  full - bodied .  Great ! "
    #         → "This wine , full - bodied . Great ! "
    s = re.sub(" +", " ", s)
    return s


# 全レビューに前処理を適用
text_data = [pad_punctuation(x) for x in filtered_data]


In [97]:
# 前処理後の同じレビューを確認 (句読点の前後にスペースが入っているはず)
example_data = text_data[25]
example_data


'wine review : US : California : Pinot Noir : Oak and earth intermingle around robust aromas of wet forest floor in this vineyard - designated Pinot that hails from a high - elevation site . Small in production , it offers intense , full - bodied raspberry and blackberry steeped in smoky spice and smooth texture . '

In [98]:
# TextVectorization に渡すため、tf.data.Dataset に変換しバッチ化 + シャッフル
text_ds = (
    tf.data.Dataset.from_tensor_slices(text_data)
    .batch(BATCH_SIZE)
    .shuffle(1000)
)


In [99]:
# 単語列を整数 ID 列に変換する TextVectorization 層を定義。
# 学習可能な「語彙テーブル」を持つ層で、後で adapt() を呼ぶと実データから語彙を学習する。
vectorize_layer = layers.TextVectorization(
    # 入力テキストに適用する標準化 (前処理) の指定。受け入れる値:
    #   - None                              … 何もしない
    #   - "lower"                           … 小文字化のみ
    #   - "lower_and_strip_punctuation"     … 小文字化 + 句読点除去 (デフォルト)
    #   - 任意の callable                   … 自前の前処理関数を渡せる
    #
    # ここで "lower" にしている理由: 句読点は前のセルの pad_punctuation で
    # 既に独立した「単語」(前後にスペース付き) として整形済み。"lower_and_strip_punctuation"
    # を選んでしまうとせっかく分けた句読点トークンが消えてしまうので、小文字化だけに留める。
    standardize="lower",

    # 語彙の上限件数。adapt() のときにデータ全体で単語頻度をカウントし、
    # 上位 max_tokens 語だけを正規の語彙として登録する。残り (低頻度な単語) は
    # 全部 [UNK] (ID = 1) にまとめられる。
    #
    # 内部的に 2 つの ID が予約済み:
    #   ID 0 = パディング (短い文を output_sequence_length まで埋めるための「空白」)
    #   ID 1 = [UNK]      (語彙外トークンを表す特殊トークン)
    # したがって実質的に使える「普通の単語」は max_tokens - 2 個。
    max_tokens=VOCAB_SIZE,

    # 出力の形式。代表的な選択肢:
    #   - "int"       … 単語 1 個 → 整数 ID 1 個 (系列の順序を保つ。RNN/Transformer 系で使う)
    #   - "multi_hot" … 文中に出てきた単語を 0/1 のベクトルで表す (Bag-of-Words)
    #   - "count"     … 単語ごとの出現回数のベクトル
    #   - "tf_idf"    … TF-IDF のベクトル
    # GPT は次トークン予測なので順序が必須 → "int" を選ぶ。
    output_mode="int",

    # 出力系列の固定長。short < N+1 は 0 (パディング) で埋め、long > N+1 は切り捨てる。
    # こうすることで、文の長さがバラバラでも 1 つのバッチに固定形状で揃えられる。
    #
    # +1 している理由: 自己回帰モデルの訓練データは「1 トークンずらした入力/ターゲット」が必要。
    #   tokens (長さ N+1) → x = tokens[:N], y = tokens[1:]   (どちらも長さ N)
    # この切り出しは後段の prepare_inputs() で行う。
    output_sequence_length=MAX_LEN + 1,
)


In [100]:
# TextVectorization 層に「語彙テーブル」を覚えさせる処理。
#
# adapt() は、通常の層の fit() に相当する「データから何かを覚える」メソッド。
# ただし勾配で重みを更新するわけではなく、統計的な集計を行うだけ。
# 例: Normalization 層なら平均と分散、TextVectorization なら単語の頻度集計。
#
# adapt(text_ds) を呼ぶと、内部で次の処理が走る:
#   1. text_ds を 1 周走査 (= 13 万件のレビュー全部を見る)
#   2. 各文を「半角スペース区切り」で単語に分割し、小文字化 (standardize="lower" のため)
#   3. 全データでの単語の出現回数をカウント
#      例: {"the": 95000, ",": 90000, "wine": 80000, ...}
#   4. 頻度上位 (max_tokens - 2) 個を採用 (ID 0/1 は予約のため -2)
#   5. 頻度順に ID を割り振って語彙テーブルを構築:
#        ID 0: ""        (パディング、予約)
#        ID 1: "[UNK]"   (語彙外、予約)
#        ID 2: "the"     ← 最頻語
#        ID 3: ","
#        ID 4: "a"
#        ...
#        ID 9999: "vibrant"  ← 9998 位の単語
#
# adapt() 完了後は vectorize_layer("wine review") のように呼べば
# テーブル参照で [25, 89] のような整数 ID 列に変換できる。
vectorize_layer.adapt(text_ds)

# 学習された語彙テーブルから単語リスト (ID 順) を取り出す。
# vocab[ID] でその ID に対応する単語が引ける逆引き辞書として後で使う
# (テキスト生成時に「予測した ID → 実際の単語」を逆変換する)。
vocab = vectorize_layer.get_vocabulary()


In [101]:
# 語彙の先頭 10 件を確認 (0 = パディング、1 = [UNK] が予約済み)
for i, word in enumerate(vocab[:10]):
    print(f"{i}: {word}")


0: 
1: [UNK]
2: :
3: ,
4: .
5: and
6: the
7: wine
8: a
9: of


In [102]:
# 先ほどの example_data を整数 ID 列に変換した結果を確認
example_tokenised = vectorize_layer(example_data)
print(example_tokenised.numpy())


[   7   10    2   20    2   29    2   43   62    2   55    5  243 4145
  453  634   26    9  497  499  667   17   12  142   14 2214   43   25
 2484   32    8  223   14 2213  948    4  594   17  987    3   15   75
  237    3   64   14   82   97    5   74 2633   17  198   49    5  125
   77    4    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0]


## 3. Create the Training Set <a name="create"></a>

In [103]:
# 自己回帰モデルの訓練データを作る。1 つの文から「入力 x」と「ターゲット y」を
# 1 トークンずらして切り出すのがポイント。
#
# === 何をやっているか ===
# GPT は「これまでの単語列を見て、次の 1 単語を予測する」を学習する。
# 1 つの文 (N+1 トークン) から、x と y を以下のようにずらして取り出すと、
# 位置 i において「x[i] (今のトークン) を見て y[i] (次のトークン) を当てる」
# という N 個の独立した予測タスクが手に入る。
#
#   元のトークン列 (N+1 個): [t0, t1, t2, ..., tN-1, tN]
#                              ↓                       ↑捨てる
#   x (N 個):              [t0, t1, t2, ..., tN-1]
#                              ↑捨てる                 ↓
#   y (N 個):                  [t1, t2, t3, ...,  tN]
#
# 例: "wine review : us :" (5 トークン、MAX_LEN=4)
#   位置 i  | x[i] (入力)   | y[i] (正解)   | 学習する関係
#   --------|----------------|----------------|---------------------------------
#   0       | "wine"         | "review"       | "wine" の次は "review"
#   1       | "review"       | ":"            | "wine review" の次は ":"
#   2       | ":"            | "us"           | "wine review :" の次は "us"
#   3       | "us"           | ":"            | "wine review : us" の次は ":"
#
# 「位置 i で x[0..i] までしか見られない」のは TransformerBlock の因果マスクが
# 保証している。これにより、1 文を 1 回フォワードするだけで N 個の訓練サンプルが
# 並列に学習できる (= トランスフォーマの強み)。
def prepare_inputs(text):
    text = tf.expand_dims(text, -1)               # (batch,) → (batch, 1) に整形
    tokenized_sentences = vectorize_layer(text)   # 文字列 → 整数 ID 列 (長さ MAX_LEN+1)
    x = tokenized_sentences[:, :-1]               # 入力: 末尾の 1 トークンを捨てる
    y = tokenized_sentences[:, 1:]                # ターゲット: 先頭の 1 トークンを捨てる
    return x, y


# text_ds の各バッチに prepare_inputs を適用 (lazy 評価)
train_ds = text_ds.map(prepare_inputs)


In [104]:
# train_ds から 1 バッチ取り出して中身を確認
example_input_output = train_ds.take(1).get_single_element()


In [105]:
# バッチ内 1 件目の入力 (整数 ID 列、長さ MAX_LEN=80)
example_input_output[0][0]


<tf.Tensor: shape=(80,), dtype=int64, numpy=
array([   7,   10,    2,   85,    2,  332,   85,    2,  777,    2,  178,
         36,   26,   39, 4150,  552,    3,   11,    8,  108,    9,   94,
        740, 1899,   17,    4,   52,  522,    3, 1338,   28, 1500,    6,
         60,    3,   80,   12,  777,   32,    6, 3070,  542,    1,  592,
        245,    9,  138,   24,  132,    5,  278,    4,    8,   73,   31,
         13,   56,   33, 2865,    4,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0])>

In [106]:
# 同じ位置の出力 (入力を 1 トークン右にシフトしたもの = 次トークン)
example_input_output[1][0]


<tf.Tensor: shape=(80,), dtype=int64, numpy=
array([  10,    2,   85,    2,  332,   85,    2,  777,    2,  178,   36,
         26,   39, 4150,  552,    3,   11,    8,  108,    9,   94,  740,
       1899,   17,    4,   52,  522,    3, 1338,   28, 1500,    6,   60,
          3,   80,   12,  777,   32,    6, 3070,  542,    1,  592,  245,
          9,  138,   24,  132,    5,  278,    4,    8,   73,   31,   13,
         56,   33, 2865,    4,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0])>

## 5. Create the causal attention mask function <a name="causal"></a>

In [107]:
# === 因果マスク (causal attention mask) ===
#
# 何のためのもの:
#   GPT は「これまでの単語列を見て次の単語を予測する」モデル。学習時には 1 つの文を
#   1 回フォワードするだけで「位置 0 → 位置 1 を予測」「位置 1 → 位置 2 を予測」…の
#   N 個のタスクを並列に学習する (前のセル prepare_inputs 参照)。
#
#   ここで困るのが、トランスフォーマのアテンションは「全位置から全位置に同時に」
#   情報をやり取りするということ。何もしないと、位置 1 で「次は cat」と予測しようと
#   している瞬間に、位置 2 や 3 にある正解の "cat" や "sat" の単語ベクトルが
#   見えてしまう ── これでは答えを覗き見しているだけで何も学習にならない。
#
#   そこで「位置 i は位置 j ≤ i にしか注意を向けられない」という制約を、
#   アテンション重みに掛けるマスクで実現する。これが因果マスク。
#
# マスクの中身 (例: 長さ 5):
#
#         j=0  j=1  j=2  j=3  j=4
#   i=0    1    0    0    0    0    ← 位置 0 は位置 0 (自分) しか見られない
#   i=1    1    1    0    0    0    ← 位置 1 は位置 0, 1 が見られる
#   i=2    1    1    1    0    0    ← 位置 2 は位置 0, 1, 2 が見られる
#   i=3    1    1    1    1    0    ← 位置 3 は位置 0, 1, 2, 3 が見られる
#   i=4    1    1    1    1    1    ← 位置 4 は全部 (= 自分まで) 見られる
#
#   下三角に 1、それ以外は 0。実際のアテンション計算では、softmax を取る前に
#   「マスク = 0 の場所のスコアを -infinity に置き換える」という形で使われる。
#   softmax(-inf) = 0 になるので、対応する位置に注意が向かなくなる。
#
# 引数:
#   batch_size … バッチサイズ (バッチ全件にコピー展開する用)
#   n_dest     … 「注意を払う側 (destination/query)」の系列長 = 行数
#   n_src      … 「注意を払われる側 (source/key)」の系列長 = 列数
#                自己アテンションでは n_dest == n_src になる
#   dtype      … 出力マスクの型 (TransformerBlock 側では tf.bool で呼ばれる)
def causal_attention_mask(batch_size, n_dest, n_src, dtype):
    # 行インデックス i を列ベクトル shape=(n_dest, 1) で作る
    i = tf.range(n_dest)[:, None]
    # 列インデックス j を行ベクトル shape=(n_src,) で作る
    j = tf.range(n_src)
    # ブロードキャストで shape=(n_dest, n_src) の真偽値テンソルを生成。
    # n_dest == n_src の場合、式は `i >= j` に等しく、下三角に True が立つ。
    # n_dest != n_src の一般化のため `- n_src + n_dest` の補正項が入っている
    # (このノートブックでは常に n_dest == n_src なので補正項は 0)。
    m = i >= j - n_src + n_dest
    # 真偽値 → 指定 dtype (bool / int / float など) に変換
    mask = tf.cast(m, dtype)
    # アテンション層が期待する形状 [batch, n_dest, n_src] に合わせるため、
    # 先頭にバッチ次元を追加
    mask = tf.reshape(mask, [1, n_dest, n_src])
    # tf.tile に渡す乗数を作る: [batch_size, 1, 1]
    # → バッチ次元方向に batch_size 倍にコピーし、他の次元はそのまま
    mult = tf.concat(
        [tf.expand_dims(batch_size, -1), tf.constant([1, 1], dtype=tf.int32)], 0
    )
    return tf.tile(mask, mult)


# 動作確認: 10x10 のマスクを生成して可視化。
# np.transpose しているのは表示の都合 (お好み)。下三角 (1 が並ぶ) になっているのが
# 確認できれば OK。実際の学習時はこれを TransformerBlock 内で seq_len に合わせて
# 動的生成し、MultiHeadAttention の attention_mask 引数として渡している。
np.transpose(causal_attention_mask(1, 10, 10, dtype=tf.int32)[0])


array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]], dtype=int32)

## 6. Create a Transformer Block layer <a name="transformer"></a>

In [108]:
# TransformerBlock = マルチヘッドアテンション + FFN + 残差接続 + LayerNorm を 1 つにまとめた層。
#
# ====================================================================
# 内部で使う Keras 層 (5 種類) の役割
# ====================================================================
#
# (1) layers.MultiHeadAttention(num_heads, key_dim, output_shape)
# --------------------------------------------------------------------
#   トランスフォーマの心臓部。アテンション機構そのもの。
#
#   ■ Q, K, V への線形変換とは
#     各位置の埋め込みベクトル x (= EMBEDDING_DIM 次元) を、ヘッドごとに 3 つの
#     学習可能な重み行列で別々に線形変換 (= 行列を掛けるだけ):
#         Q = x · W_Q   ← クエリ:   「何を探しているか」のフィルター
#         K = x · W_K   ← キー:     「どんな情報を持っているか」のラベル
#         V = x · W_V   ← バリュー: 「参照されたときに渡す情報」
#     W_Q / W_K / W_V は学習で獲得する重み行列。同じ x から 3 つの違う役割の
#     ベクトルが作られる。検索エンジンのアナロジー: 検索窓に入れる言葉 = Q、
#     ページのタイトル/メタ情報 = K、ページ本文 = V。
#
#   ■ 内積で関連度を測る (Q · K^T)
#     2 つのベクトルの「関連度合い」は、**対応する要素を掛けて全部足した値**
#     (= 内積、dot product) で測る:
#
#         a · b = a1·b1 + a2·b2 + a3·b3 + ... + aN·bN
#
#     幾何学的には a · b = |a| · |b| · cos(θ) で、ベクトルのなす角 θ を反映:
#       - 同じ向き (θ ≈ 0°)   → cos = 1   → 内積が大きい (関連度高)
#       - 直交  (θ = 90°)     → cos = 0   → 内積はゼロ (無関係)
#       - 真逆  (θ = 180°)    → cos = -1  → 内積が負に大きい
#
#     アテンションでは Q (5 単語 × 256 次元) と K (5 単語 × 256 次元) の
#     行列まるごとの内積 Q · K^T で、全位置ペアの関連度を 5×5 行列として一発計算:
#
#                  K_wine  K_review  K_:  K_germany  K_:
#         Q_wine     3.2     1.8     0.4    2.1      0.5
#         Q_review   1.6     4.1     0.8    1.9      0.7    ← [i][j] = Q[i] · K[j]
#         Q_:        0.5     0.9     5.3    1.2      1.5      (位置 i から位置 j への
#         Q_germany  2.0     1.8     1.1    3.8      0.9       関連度スコア)
#         Q_:        0.4     0.8     1.6    3.1      4.7
#
#     内積を使う理由 (コサイン類似度などではなく):
#       - 計算が GPU で並列化しやすい (行列乗算 1 発)
#       - 学習で微分可能
#       - 「強くマッチしたい単語の K を大きくする」というベクトルの長さ戦略が取れる
#         (コサイン類似度は長さの情報を捨てるので、この自由度がない)
#
#   ■ アテンション重みとは
#     「位置 i は位置 j にどれだけ注意を向けるか」を表す数値 (合計 1 の確率分布)。
#     上の関連度スコア行列を確率分布化したもの。計算手順:
#         1. Q[i] と K[j] の内積 → 関連度スコア (上の表):
#              scores[i, j] = Q[i] · K[j]
#         2. sqrt(key_dim) でスケーリング (内積が大きすぎて softmax が極端な分布に
#            なるのを防ぐため):
#              scaled[i, j] = scores[i, j] / sqrt(key_dim)
#         3. 因果マスク適用 (未来位置のスコアを -inf にする → 前のセルで定義した
#            causal_attention_mask を attention_mask 引数で渡している)
#         4. softmax で確率分布化:
#              weights[i, j] = exp(scaled[i, j]) / Σ_k exp(scaled[i, k])
#     weights[i, j] が「位置 i から位置 j への注意の重み」。具体例: 文末で
#     "wine review : germany :" の次を予測するとき、weights[4, :] が
#     [0.01, 0.02, 0.05, 0.85, 0.07] のように "germany" (位置 3) に 85% 注目する
#     ような分布が学習で得られる。この値が後段の print_probs で青ハイライトされる。
#
#   ■ 最終出力
#     アテンション重みで V を加重平均:
#         output[i] = Σ_j weights[i, j] · V[j]
#     注意度の高い位置の V を強く取り込んだベクトルが出る。これを num_heads 個
#     並列に計算 → 結合 → output_shape 次元に線形射影して返す。
#
# (2) layers.Dropout(rate)
# --------------------------------------------------------------------
#   過学習防止の正則化。訓練時のみ入力の rate% (例: 10%) をランダムに 0 にし、
#   残りを 1/(1-rate) 倍してスケールを保つ。推論時は何もしない (恒等関数)。
#   GPT 系では rate=0.1 がデフォルト。
#
# (3) layers.LayerNormalization(epsilon=1e-6)
# --------------------------------------------------------------------
#   各サンプル・各位置で「特徴次元方向」に平均と分散を計算し、平均 0・分散 1 に
#   正規化したあと、学習可能なスケール (gamma) とシフト (beta) を掛ける。
#   学習の安定化に寄与する。
#   BatchNorm との違い: バッチ軸ではなく特徴軸方向に正規化するのでバッチサイズに
#   依存せず、訓練/推論で挙動が同じ → 系列モデル (RNN/Transformer) で定番。
#   epsilon は分母が 0 になるのを防ぐ小さな定数。
#
# (4) layers.Dense(units, activation=None)
# --------------------------------------------------------------------
#   全結合層 = 線形変換 + 活性化関数: y = activation(W · x + b)。
#   ここでは 2 つの Dense を直列に並べて「FFN (Feed-Forward Network)」を構成:
#       ffn_1: Dense(ff_dim, relu)  ← 拡張 + 非線形変換
#       ffn_2: Dense(embed_dim)      ← 圧縮 (活性化なし)
#   アテンションが「位置間のやり取り」を担うのに対し、FFN は「各位置で独立した
#   特徴変換」を担う。中間次元 ff_dim は通常 embed_dim の 4 倍にすることが多い
#   (このノートブックでは同じ 256 で簡略化)。
#
# ====================================================================
# call() の流れ (= データの流れ)
# ====================================================================
#
#   inputs ───┬───→ MultiHeadAttention(因果マスク) ───→ Dropout ───┐
#             │                                                       │
#             └──────────────── 残差 + LayerNorm ←─────────────────────┘
#                                      │
#                                      ↓
#                       ┌── Dense(ff_dim, relu) → Dense(embed_dim) → Dropout ─┐
#                       │                                                        │
#                       └────────────── 残差 + LayerNorm ←──────────────────────┘
#                                              │
#                                              ↓
#                                       (output, attention_scores)
#
#   残差接続 (inputs + ...) は、深いネットワークでも勾配が流れるようにする
#   重要な仕組み (ResNet 由来)。「元の入力に修正を加える」発想で学習が安定する。
class TransformerBlock(layers.Layer):
    def __init__(self, num_heads, key_dim, embed_dim, ff_dim, dropout_rate=0.1):
        super(TransformerBlock, self).__init__()
        self.num_heads = num_heads
        self.key_dim = key_dim
        self.embed_dim = embed_dim
        self.ff_dim = ff_dim
        self.dropout_rate = dropout_rate
        # マルチヘッドアテンション層 (Q/K/V 内部生成、因果マスク適用、加重平均まで全部やってくれる)
        self.attn = layers.MultiHeadAttention(
            num_heads, key_dim, output_shape=embed_dim
        )
        self.dropout_1 = layers.Dropout(self.dropout_rate)
        self.ln_1 = layers.LayerNormalization(epsilon=1e-6)
        # FFN: Dense ×2 で「拡張 → ReLU → 圧縮」の 2 層 MLP
        self.ffn_1 = layers.Dense(self.ff_dim, activation="relu")
        self.ffn_2 = layers.Dense(self.embed_dim)
        self.dropout_2 = layers.Dropout(self.dropout_rate)
        self.ln_2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size = input_shape[0]
        seq_len = input_shape[1]
        # 各バッチ・各系列長に合わせて因果マスクを動的生成
        causal_mask = causal_attention_mask(
            batch_size, seq_len, seq_len, tf.bool
        )
        # Self-attention: query = key = value = inputs (= Q/K/V を全部 inputs から作る)
        attention_output, attention_scores = self.attn(
            inputs,
            inputs,
            attention_mask=causal_mask,
            return_attention_scores=True,   # 可視化用にアテンションスコアも返す
        )
        attention_output = self.dropout_1(attention_output)
        out1 = self.ln_1(inputs + attention_output)   # 残差接続 + LayerNorm
        # フィードフォワード経路
        ffn_1 = self.ffn_1(out1)
        ffn_2 = self.ffn_2(ffn_1)
        ffn_output = self.dropout_2(ffn_2)
        # もう一度残差接続 + LayerNorm
        return (self.ln_2(out1 + ffn_output), attention_scores)

    def get_config(self):
        # モデル保存/ロード時にカスタム層を再構築するためのコンフィグ
        config = super().get_config()
        config.update(
            {
                "key_dim": self.key_dim,
                "embed_dim": self.embed_dim,
                "num_heads": self.num_heads,
                "ff_dim": self.ff_dim,
                "dropout_rate": self.dropout_rate,
            }
        )
        return config


## 7. Create the Token and Position Embedding <a name="embedder"></a>

アテンションは順序を意識しません (キーとクエリのドット積はすべての位置ペアで並列に計算されるため)。例えば次の 2 文はアテンション層から見ると区別できません:

- *The dog looked at the boy* … (吠えた?)
- *The boy looked at the dog* … (微笑んだ?)

この問題を解消するため、トークン埋め込みに **位置埋め込み** を加算します。

```
最終的な埋め込み = TokenEmbedding(token_id) + PositionEmbedding(position)
```

原論文 (Vaswani et al., 2017) では sin/cos の三角関数による固定位置エンコーディングが使われましたが、GPT では学習可能な `Embedding` 層を使うのが標準です (このノートブックもそれに従う)。


In [109]:
# トークン埋め込み + 位置埋め込みの加算層
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, max_len, vocab_size, embed_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.max_len = max_len
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        # トークン ID → ベクトル
        self.token_emb = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim
        )
        # 位置 (0, 1, 2, ..., max_len-1) → ベクトル
        self.pos_emb = layers.Embedding(input_dim=max_len, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions   # 加算するだけ (連結ではない)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "max_len": self.max_len,
                "vocab_size": self.vocab_size,
                "embed_dim": self.embed_dim,
            }
        )
        return config


## 8. Build the Transformer model <a name="transformer_decoder"></a>

In [110]:
# === GPT モデル全体を組み立てる (書籍 図 9-9 相当) ===
#
# アーキテクチャ:
#
#   Input (整数 ID 列)
#     ↓
#   TokenAndPositionEmbedding   ← トークン埋め込み + 位置埋め込み
#     ↓
#   TransformerBlock            ← 本来は 12 層積むが、簡略化のため 1 層のみ
#     ↓
#   Dense(VOCAB_SIZE, softmax)  ← 各位置で次トークンの確率分布
#     ↓
#   Output (語彙分布) + attention_scores
#
# ----------------------------------------------------------------------
# 「本来は 12 層積む」とは: TransformerBlock の縦積み
# ----------------------------------------------------------------------
#
# TransformerBlock は入力ベクトルと同じ形のベクトルを返すので、出力を次のブロックの
# 入力として連結できる:
#
#     x ─→ TransformerBlock #1 ─→ TransformerBlock #2 ─→ ... ─→ TransformerBlock #N ─→ Dense
#
# 各ブロックは独立した重み (W_Q, W_K, W_V, FFN 等) を持つ。コードで書くと:
#
#     for _ in range(N):
#         x, _ = TransformerBlock(...)(x)
#
# 多層化のうれしさ:
#   ブロックを積むほど「より抽象的・複雑な関係」を学習できる傾向がある (interpretability
#   研究の後付け解釈)。CNN で「浅い層 = エッジ、深い層 = 物体」が起きるのと似た階層化。
#     - 浅い層 (1〜3 段目)   … 表面パターン (句読点の出現位置、地名の後の語など)
#     - 中間層 (4〜8 段目)   … 構文・局所文脈 (主語-動詞の一致、名詞句の境界)
#     - 深い層 (9〜12 段目)  … 意味/世界知識 (ドイツの mosel 地方は riesling を作る等)
#
# 実際の GPT モデルのブロック数:
#     - Transformer (原論文 2017)    … encoder 6 + decoder 6
#     - GPT-1 (2018)                 … 12
#     - GPT-2 small / med / lrg / XL … 12 / 24 / 36 / 48
#     - GPT-3 (2020)                 … 96
#     - LLaMA-2 7B / 13B / 70B       … 32 / 40 / 80
#     - GPT-4 (2023)                 … 非公開 (100+ と推測)
#
# このノートブックで 1 層だけにしている理由:
#   1. 構造を理解するには 1 層で十分 (12 層あっても本質は「同じブロックの縦積み」だけ)
#   2. 学習時間: Colab T4 で 1 ブロック × 5 エポックが数分。12 ブロックなら単純計算で 12 倍
#   3. GPU メモリ: 1 ブロックで百万単位のパラメータ + 勾配 + Adam の状態 → 12 倍は無料 Colab だときつい
#   4. 生成の質より「仕組みを学ぶ」のが目的。1 層でも書籍 9.2.9 のような文は出てくる
inputs = layers.Input(shape=(None,), dtype=tf.int32)                          # 可変長の整数 ID 列
x = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBEDDING_DIM)(inputs)     # 埋め込み (トークン + 位置)
x, attention_scores = TransformerBlock(                                       # トランスフォーマブロック (1 個だけ)
    N_HEADS, KEY_DIM, EMBEDDING_DIM, FEED_FORWARD_DIM
)(x)
outputs = layers.Dense(VOCAB_SIZE, activation="softmax")(x)                   # 各位置で次トークンの確率分布
gpt = models.Model(inputs=inputs, outputs=[outputs, attention_scores])         # 2 出力 (語彙分布 + アテンション)
# 損失は語彙分布側にだけかける (アテンション側は None)。
# SparseCategoricalCrossentropy: ターゲットが「整数 ID」のときに使うクロスエントロピー損失
# (one-hot 化を内部でやってくれる)
gpt.compile("adam", loss=[losses.SparseCategoricalCrossentropy(), None])


In [111]:
gpt.summary()   # モデル構造とパラメータ数を表示


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_3  │ (None, None, 256)      │     2,580,480 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_3             │ [(None, None, 256),    │       658,688 │
│ (TransformerBlock)              │ (None, 2, None, None)] │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, None, 10000)    │     2,570,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,809,168 (22.16 MB)

 Trainable params: 5,809,168 (22.16 MB)

 Non-trainable params: 0 (0.00 B)

In [112]:
# LOAD_MODEL=True なら保存済みモデルを再ロードして学習をスキップ
if LOAD_MODEL:
    # model.load_weights('./models/model')
    gpt = models.load_model("./models/gpt.keras", compile=True)


## 9. Train the Transformer <a name="train"></a>

学習時のセットアップとして、3 つのコールバックを使います:

1. **`TextGenerator`** … 自前のカスタム `callbacks.Callback`。毎エポック終了時に「wine review」というプロンプトから 80 トークンを生成してログ表示する。学習が進むにつれて出力がどう改善されるかを観察できる
2. **`ModelCheckpoint`** … エポックごとに重みを `./checkpoint/checkpoint.weights.h5` に保存。途中で中断しても再開できる
3. **`TensorBoard`** … `./logs` 配下に学習ログを出力。Colab 上で `%load_ext tensorboard` & `%tensorboard --logdir ./logs` で可視化可能

`TextGenerator.generate()` の `sample_from(probs, temperature)` は、`p ← p^(1/T) / Σp^(1/T)` で確率分布を「鋭く/平坦に」してからサンプリングします (`T < 1` で鋭く決定論的に、`T > 1` で平坦で多様に)。


In [113]:
# エポックごとにテキスト生成テストを行うカスタムコールバック
class TextGenerator(callbacks.Callback):
    def __init__(self, index_to_word, top_k=10):
        self.index_to_word = index_to_word   # ID → 単語の対応 (語彙リスト)
        self.word_to_index = {                # 逆方向: 単語 → ID
            word: index for index, word in enumerate(index_to_word)
        }

    def sample_from(self, probs, temperature):
        # temperature サンプリング: p ← p^(1/T) / Σp^(1/T)
        # T < 1 で分布が鋭く (決定論的)、T > 1 で平坦 (多様) になる
        probs = probs ** (1 / temperature)
        probs = probs / np.sum(probs)
        return np.random.choice(len(probs), p=probs), probs

    def generate(self, start_prompt, max_tokens, temperature):
        # プロンプトをトークン ID 列に変換 (未知語は 1 = [UNK])
        start_tokens = [
            self.word_to_index.get(x, 1) for x in start_prompt.split()
        ]
        sample_token = None
        info = []   # 各ステップのプロンプト/確率/アテンションを記録
        # max_tokens に達するか、終端トークン (0) が出るまで生成
        while len(start_tokens) < max_tokens and sample_token != 0:
            x = np.array([start_tokens])
            y, att = self.model.predict(x, verbose=0)
            # 最後の位置の出力分布から次トークンをサンプリング
            sample_token, probs = self.sample_from(y[0][-1], temperature)
            info.append(
                {
                    "prompt": start_prompt,
                    "word_probs": probs,
                    "atts": att[0, :, -1, :],   # 最後の位置のアテンション
                }
            )
            start_tokens.append(sample_token)
            start_prompt = start_prompt + " " + self.index_to_word[sample_token]
        print(f"\ngenerated text:\n{start_prompt}\n")
        return info

    def on_epoch_end(self, epoch, logs=None):
        # 各エポック末に「wine review」プロンプトで生成して進捗を観察
        self.generate("wine review", max_tokens=80, temperature=1.0)


In [114]:
# エポックごとに重みを保存するチェックポイント
model_checkpoint_callback = callbacks.ModelCheckpoint(
    filepath="./checkpoint/checkpoint.weights.h5",
    save_weights_only=True,
    save_freq="epoch",
    verbose=0,
)

# TensorBoard 用のログ出力
tensorboard_callback = callbacks.TensorBoard(log_dir="./logs")

# 訓練中の生成テストを行うコールバック
text_generator = TextGenerator(vocab)


In [115]:
# 学習開始 (Colab T4 GPU で 1 エポックあたり数分)
gpt.fit(
    train_ds,
    epochs=EPOCHS,
    callbacks=[model_checkpoint_callback, tensorboard_callback, text_generator],
)


Epoch 1/5
4060/4060 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 2.6018
generated text:
wine review : us : california : petite sirah : the vineyard is sangiovese and [UNK] savory , all cabs , as good now . the tannins have this is a cherry - berry , cola and dark chocolate flavors . the [UNK] , barbecue is definitely a wine to match for a few more years in the years of aging . 

4060/4060 ━━━━━━━━━━━━━━━━━━━━ 166s 39ms/step - loss: 2.2456
Epoch 2/5
4059/4060 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 1.9771
generated text:
wine review : portugal : beira atlantico : [UNK] : the honor of [UNK] means close to [UNK] : soft , rich wine concentrates lemon and apricot fruits . 

4060/4060 ━━━━━━━━━━━━━━━━━━━━ 111s 27ms/step - loss: 1.9575
Epoch 3/5
4060/4060 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 1.8982
generated text:
wine review : us : california : cabernet sauvignon : though a little heavy , and so drink up . it ' s ripe , the is a bit sugary , with a hint of pineapple , fig and finishes sh

In [116]:
# 学習完了後のモデル全体を Keras 3 形式 (.keras) で保存
gpt.save("./models/gpt.keras")


# 3. Generate text using the Transformer

**書籍 §9.2.9「GPT の分析」に対応するセクション**です。訓練済みモデルを使って 2 つの分析を行います:

### §9.2.9.1 テキスト生成

`temperature` パラメータの効果を観察します:

- `temperature = 1.0` … 分布のままサンプリングする (多様だが時々おかしい単語が出る)
- `temperature = 0.5` … 分布を鋭くしてからサンプリング (決定論的に近く、最頻語に偏る → 自然な文だが画一的)

### §9.2.9.2 アテンション可視化

`print_probs(info, vocab)` が以下を出力します:

- **HTML ハイライト**: アテンション重みの平均を背景色 (青の濃淡) で表示。「ドイツ → mosel/rheingau」のように、関連する位置にどれだけ注意が向いているかが視覚的にわかる
- **トップ K 単語の確率分布**: 各位置でモデルが次に何を予測しようとしているかを上位 5 件で表示 (図 9-11 と同じ)

入力プロンプトの言葉を変えて、モデルが文中のどの単語に注目するかを観察してみてください。


In [117]:
# 生成過程の info を可視化する関数 (書籍 図 9-11 相当)。
# 2 種類の可視化を 1 回の生成プロセスについて出力する:
#
#   ① アテンション重みを LightSkyBlue の濃淡で背景色ハイライト表示
#   ② 次トークン候補のトップ K (デフォルト 5) を確率付きでテキスト表示
#
# ─────────────────────────────────────────────────────────────────────
# ① 背景色ハイライトの仕組み
# ─────────────────────────────────────────────────────────────────────
# 色そのもの: rgba(135, 206, 250, α) = LightSkyBlue (薄い水色)。α が透明度。
#   - α = 1.0 → 不透明 (濃い水色)         → アテンションが強い
#   - α = 0.0 → 完全透明 (背景の白が見える) → ほぼ無視されている
#
# α の計算: 「ヘッド平均したアテンション重み ÷ 全単語中の最大値」 で 0〜1 に正規化。
#
# 例: "wine review : germany" の次を予測するとき、i["atts"] は shape (num_heads, seq_len)
# のテンソルで以下のような値:
#
#                wine  review   :   germany
#     ヘッド0  [ 0.05,  0.10, 0.05,  0.80 ]
#     ヘッド1  [ 0.10,  0.15, 0.20,  0.55 ]
#
# Step 1: ヘッド方向に平均化 → np.mean(i["atts"], axis=0)
#         平均値      [ 0.075, 0.125, 0.125, 0.675 ]
#
# Step 2: 最大値 (= germany の 0.675) で割って正規化
#         α (= 透明度) [ 0.111, 0.185, 0.185, 1.000 ]
#
# Step 3: 各単語に <span style="background-color:rgba(135,206,250, α)"> を付けて HTML 描画
#         → "wine" 薄い / "review" やや薄い / ":" やや薄い / "germany" 濃い水色
#
# 注意:
#   - 正規化のため「色のレンジ (一番薄い〜一番濃い)」は常に同じになる。アテンションが
#     均一でも 1 単語に集中していても、相対的な分布だけを表示する。絶対的なアテンション
#     値を表しているわけではない (注目度の強さ自体を比較するには値そのものを見る必要がある)
#   - 2 つのヘッドを平均化して 1 枚に統合しているので、ヘッドごとの違いは見えなくなる
#
# ─────────────────────────────────────────────────────────────────────
# ② トップ K 候補表示
# ─────────────────────────────────────────────────────────────────────
# i["word_probs"] には softmax 後の全語彙 (10000 語) の確率分布が入っている。
# np.sort / np.argsort で確率の降順に並べ替え、上位 top_k 件を「単語名: 確率%」形式で出力。
def print_probs(info, vocab, top_k=5):
    for i in info:
        # ① アテンション重みを背景色でハイライトした HTML を生成
        highlighted_text = []
        for word, att_score in zip(
            i["prompt"].split(), np.mean(i["atts"], axis=0)   # ヘッド方向に平均
        ):
            highlighted_text.append(
                '<span style="background-color:rgba(135,206,250,'
                + str(att_score / max(np.mean(i["atts"], axis=0)))   # 最大値で割って α (0〜1) に正規化
                + ');">'
                + word
                + "</span>"
            )
        highlighted_text = " ".join(highlighted_text)
        display(HTML(highlighted_text))

        # ② 次トークン候補のトップ K を確率付きで表示
        word_probs = i["word_probs"]
        p_sorted = np.sort(word_probs)[::-1][:top_k]    # 確率の降順
        i_sorted = np.argsort(word_probs)[::-1][:top_k] # 対応する語彙 ID の降順
        for p, i in zip(p_sorted, i_sorted):
            print(f"{vocab[i]}:   \t{np.round(100*p,2)}%")
        print("--------\n")


In [118]:
# 米国産ワインのレビューを temperature=1.0 (多様) で生成
info = text_generator.generate(
    "wine review : us", max_tokens=80, temperature=1.0
)



generated text:
wine review : us : new york : riesling : hints of cinnamon and ripe melon lend a delicate tone to this bristling off - dry riesling . it ' s severely concentrated on the new york lemon and tangerine flavors against a plump mineral tone . on the finish meanders beautifully by cutting acidity . 



In [119]:
# イタリア産ワインを temperature=0.5 (決定論的寄り) で生成
info = text_generator.generate(
    "wine review : italy", max_tokens=80, temperature=0.5
)



generated text:
wine review : italy : lombardy : turbiana : this opens with aromas of pressed spanish broom , acacia and orchard fruit . the medium - bodied palate delivers white peach , mature yellow apple and mineral alongside fresh acidity . 



In [120]:
# ドイツ産ワインを生成し、各位置のアテンション + 確率分布を可視化 (図 9-11 相当)
info = text_generator.generate(
    "wine review : germany", max_tokens=80, temperature=0.5
)
print_probs(info, vocab)



generated text:
wine review : germany : mosel : riesling : the nose on this wine , the palate of this kabinett , the ' 09 kabinett is a showcase of mosel ' s wines , and the balance of [UNK] ' s sugar - laced oolong tea . it ' s nuanced , yet rich and concentrated , with a lingering , honeyed finish . 



::   	100.0%
,:   	0.0%
-:   	0.0%
.:   	0.0%
grosso:   	0.0%
--------



mosel:   	91.55999755859375%
rheingau:   	4.170000076293945%
rheinhessen:   	2.119999885559082%
pfalz:   	1.7100000381469727%
nahe:   	0.33000001311302185%
--------



::   	100.0%
-:   	0.0%
,:   	0.0%
and:   	0.0%
grosso:   	0.0%
--------



riesling:   	100.0%
pinot:   	0.0%
white:   	0.0%
gewürztraminer:   	0.0%
[UNK]:   	0.0%
--------



...

,:   	84.6500015258789%
finish:   	6.090000152587891%
lime:   	1.1799999475479126%
mineral:   	1.1200000047683716%
lace:   	0.7900000214576721%
--------



complex:   	23.5%
penetrating:   	20.030000686645508%
mineral:   	10.649999618530273%
honeyed:   	5.170000076293945%
long:   	4.920000076293945%
--------



finish:   	95.72000122070312%
note:   	2.2899999618530273%
,:   	0.5600000023841858%
tone:   	0.28999999165534973%
sweetness:   	0.23999999463558197%
--------



.:   	99.98999786376953%
,:   	0.009999999776482582%
that:   	0.0%
with:   	0.0%
on:   	0.0%
--------



:   	93.54000091552734%
drink:   	5.039999961853027%
it:   	0.9200000166893005%
the:   	0.2800000011920929%
a:   	0.07000000029802322%
--------

